In [34]:
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import Subset,DataLoader
import torch
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.optim as optim

In [40]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu")
device

device(type='mps')

In [3]:
transform = transforms.Compose([transforms.ToTensor(),transforms.Normalize(0.5,0.5)])
train_dataset = datasets.FashionMNIST(root="data",train=True,download=True,transform=transform)
test_dataset = datasets.FashionMNIST(root="data",train=False,download=True,transform=transform)

100%|██████████████████████████████████████████████████████████████| 26.4M/26.4M [00:07<00:00, 3.55MB/s]
100%|███████████████████████████████████████████████████████████████| 29.5k/29.5k [00:00<00:00, 172kB/s]
100%|██████████████████████████████████████████████████████████████| 4.42M/4.42M [00:04<00:00, 1.07MB/s]
100%|██████████████████████████████████████████████████████████████| 5.15k/5.15k [00:00<00:00, 7.11MB/s]


In [4]:
len(train_dataset),len(test_dataset)

(60000, 10000)

In [7]:
torch.arange(5000)

tensor([   0,    1,    2,  ..., 4997, 4998, 4999])

In [10]:
train_subset = Subset(train_dataset,torch.arange(5000))
test_subset = Subset(test_dataset,torch.arange(1000))

In [15]:
train_loader = DataLoader(train_subset,batch_size=60,shuffle=True)
test_loader = DataLoader(test_subset,batch_size=60,shuffle=False)

In [30]:
images,labels = next(iter(train_loader))
images.shape

torch.Size([60, 1, 28, 28])

In [31]:
len(train_loader)

84

### Creating NN

In [33]:
class MnistClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28,128),
            nn.ReLU(),
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Linear(64,10)
        )
    def forward(self,x):
        return self.network(x)

In [45]:
model = MnistClassifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=0.001)

In [47]:
epochs = 10
for epoch in range(epochs):
    model.train()
    for batch,(images,labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs,labels)
        loss.backward()
        optimizer.step()

        print(f"Epoch: {epoch+1}/{epochs}, Batch:{batch+1}, Loss: {loss.item():.3f}")

    model.eval()
    for images,labels in test_loader:
        outputs = model(images)
        
        break

Epoch: 1/10, Batch:1, Loss: 2.286
Epoch: 2/10, Batch:1, Loss: 2.261
Epoch: 3/10, Batch:1, Loss: 2.181
Epoch: 4/10, Batch:1, Loss: 2.146
Epoch: 5/10, Batch:1, Loss: 2.024
Epoch: 6/10, Batch:1, Loss: 2.017
Epoch: 7/10, Batch:1, Loss: 1.923
Epoch: 8/10, Batch:1, Loss: 1.810
Epoch: 9/10, Batch:1, Loss: 1.739
Epoch: 10/10, Batch:1, Loss: 1.594
